# 🧬 FlyOpt: Biyolojik Optimizasyon Motoru (Colab GPU/T4 Sürümü)

Bu notebook, **Gemini** tarafından projenizi yüksek hızlı bulut GPU'larına (T4) taşımak ve erkek sinek (Male CNS) beynini T=20 parametresiyle yepyeni biyolojik misyonlarda (Foraging/Pareto) test etmek için hazırlanmıştır.

In [ ]:
# 1. MİSYON: DONANIM VE PROJE BAĞLANTISI
# Yerel projenizi Google Drive'a yükleyin (klasör adı 'fly_op' olmalı) ve bu hücreyi çalıştırın.
from google.colab import drive
import sys
import os
import torch

drive.mount('/content/drive')
project_path = '/content/drive/MyDrive/fly_op'  # Drive'daki proje yolunuz
sys.path.append(project_path)
os.chdir(project_path)

print('GPU Durumu:')
!nvidia-smi

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nAktif Donanım: {device}')


### 2. MİSYON: HIZ TESTİ VE T=20 BİYOLOJİK MOTOR YÜKLEMESİ
Lokalde saatler süren işlemi Colab T4 GPU üzerinde toplu (batched) tensörlerle saniyeler içinde çözüyoruz.

In [ ]:
import numpy as np
from scipy import sparse
import time
from flyopt.variants.rate_brain import RateBrain, RateBrainConfig, build_subgraph_bfs, select_connected_encode_decode

# Male CNS Verilerini Yükle (Drive üzerinden)
DATA_PROCESSED = f"{project_path}/data/processed"
base_weights = sparse.load_npz(f"{DATA_PROCESSED}/malecns_adjacency.npz")
afferent = np.load(f"{DATA_PROCESSED}/malecns_afferent_indices.npy")
efferent = np.load(f"{DATA_PROCESSED}/malecns_efferent_indices.npy")

# Alt-grafı (Subgraph) oluştur
encode_full, decode_full = select_connected_encode_decode(
    base_weights, afferent, efferent, n_encode=4, n_decode=4, max_hops=6, seed=9000
)
sub_real, encode_idx, decode_idx, _ = build_subgraph_bfs(base_weights, encode_full, decode_full, 3000, seed=9000)

# T=20 MOTORU KURULUYOR (Cuda destekli)
cfg = RateBrainConfig(dim=2, n_readout=len(decode_idx), T=20, decode_scale=0.5, train_gain=True)
brain = RateBrain(sub_real, encode_idx, decode_idx, cfg, seed=42).to(device)

print("T=20 Biyolojik Motor Başarıyla GPU'ya yüklendi!")

### 3. MİSYON: BİYOLOJİK KAYNAK ARAMA (FORAGING / PARETO)
Sinek artık düz bir matematiği çözmüyor. 10.000 farklı yöne aynı anda bakarak (dev GPU paralelizmi) en yüksek enerji/ödül değerini (Pareto Front) arıyor.

In [ ]:
# GPU üzerinde devasa paralel arama (Lokalde saatler sürer, Colab'da saniyeler)
BATCH_SIZE = 10000  # Aynı anda test edilecek senaryo sayısı

# Sinek için 10 bin farklı çevresel girdiyi (Koku/Tehlike/Engeller) simüle ediyoruz
environmental_inputs = torch.randn(BATCH_SIZE, 4, device=device) 

print(f"Motor {BATCH_SIZE} çevre koşulunu aynı anda değerlendiriyor...")
start_time = time.time()

# Ağın (beynin) T=20 adım boyunca düşünmesi ve reaksiyon üretmesi
with torch.no_grad():
    biological_responses = brain(environmental_inputs)

end_time = time.time()
print(f"\n✅ İşlem Tamamlandı!")
print(f"{BATCH_SIZE} farklı senaryo (T=20 iç-döngü) GPU'da sadece {end_time - start_time:.4f} saniyede çözüldü!")
print("Elde edilen Biyolojik Yanıt Vektörürü (Pareto Dağılımı):", biological_responses.shape)